# DMRGT Phase 1 — Kaggle Training Notebook

**Spec:** §7.3 of `DMRGT-Phase1-Build-Spec.md`.

This is a thin wrapper. All real logic lives in `model.py` and `train.py`, which are either attached as the `dmrgt-code` dataset or pasted into a code cell.

**Always run with Save → Save & Run All (Commit)** — interactive sessions die on idle and lose `/kaggle/working/`.

Steps this notebook executes:
  1. Environment check (resolve dmrgt-code mount, cache attached, GPU ≥ sm_70)
  2. Cache integrity check (delegated to `train.verify_cache_hashes` so notebook and trainer cannot disagree)
  3. Import `model.py` / `train.py`
  4. Run all folds × all seeds
  5. Print a compact summary; write `predictions.parquet` and `run_manifest.json` to `/kaggle/working/`

> **Accelerator must be GPU T4 (or L4 / A100).** Kaggle's current PyTorch build does not support P100 (sm_60). The Cell 1 assertion catches this.

## Cell 1 — Environment check

In [ ]:
import os, sys, json
from pathlib import Path

def _find_dataset(slug: str) -> str:
    root = Path("/kaggle/input")
    for c in (root / slug, root / "datasets" / slug):
        if c.is_dir():
            return str(c)
    matches = [p for p in root.rglob(slug) if p.is_dir()]
    if len(matches) == 1:
        return str(matches[0])
    raise FileNotFoundError(
        f"Dataset '{slug}' not found (matches: {len(matches)}). "
        f"Attach it, then RESTART THE SESSION.\n"
        f"Contents of /kaggle/input:\n  "
        + "\n  ".join(sorted(str(p) for p in root.rglob("*"))[:40])
    )

sys.path.insert(0, _find_dataset("dmrgt-code"))

import torch
import paths

print(json.dumps(paths.describe(), indent=2))

for label, p in [("prices", paths.PRICES), ("features", paths.FEATURES),
                 ("targets", paths.TARGETS), ("hashes", paths.HASHES_JSON),
                 ("config", paths.CONFIG)]:
    assert p.exists(), f"{label} not found at {p}"

assert torch.cuda.is_available(), \
    "Enable the GPU: Settings -> Accelerator -> GPU T4"

major, minor = torch.cuda.get_device_capability(0)
gpu = torch.cuda.get_device_name(0)
print(f"[env] GPU: {gpu} (sm_{major}{minor})")
assert major >= 7, (
    f"{gpu} has CUDA capability sm_{major}{minor}, which this PyTorch build "
    f"does not support (requires sm_70+). Switch Accelerator to GPU T4 and "
    f"restart the session. NOTE: torch.cuda.is_available() returns True even "
    f"on an unsupported card, so this check is required."
)

## Cell 2 — Cache integrity check

Recompute the SHA-256 of every parquet and assert it matches `hashes.json`. A mismatch means a stale dataset version is attached — refusing to train is the only safe response (per §2.5 / §7.3 cell 2).

In [ ]:
# Cell 2 — Cache integrity check
# Delegates to train.verify_cache_hashes so the notebook and the
# trainer cannot disagree about what "verified" means.
import train
hashes = train.verify_cache_hashes(stop_on_mismatch=True)

## Cell 3 — Import model + train

The code is attached via the `dmrgt-code` dataset. Make sure the dataset is attached and your notebook's file path includes the dataset directory. If you prefer, paste the contents of `model.py` and `train.py` here instead.

In [ ]:
# Cell 1 already added the dmrgt-code dataset to sys.path. We only
# need to verify the project modules import cleanly from here.
import model
print(f"[env] model.py: {model.__file__}")
print(f"[env] train.py: {train.__file__}")

## Cell 4 — Run all folds × all seeds

In [ ]:
# This will write predictions.parquet, train_metrics.json,
# run_manifest.json, and checkpoints/ to /kaggle/working/.
#
# The trainer:
#   • reads configs.yaml via paths.CONFIG (not CWD — fixes the
#     "configs.yaml not found" error on Kaggle)
#   • fits the scaler on training dates only (§1.1)
#   • asserts no test date appears in the training set
#   • emits per-seed test predictions in the long format the local
#     eval harness consumes
#
# Expected run time on a T4: ~1–3 minutes per fold × seed
# (Phase 1 model is small — ~100K parameters).

cfg = train.load_config()   # resolves via paths.CONFIG
print(f"[env] config: "
      f"{len(cfg['universe']['tickers'])} tickers, "
      f"{len(cfg['folds'])} folds, "
      f"{len(cfg['training']['seeds'])} seeds")
train.run_all(cfg, verify_hashes=False, hashes=hashes)  # hashes from cell 2

## Cell 5 — Summary + outputs

In [ ]:
import json
from pathlib import Path

# Confirm the required outputs exist before the session dies.
out = Path("/kaggle/working")
required = ["predictions.parquet", "train_metrics.json", "run_manifest.json"]
for name in required:
    p = out / name
    assert p.exists(), f"Missing required output: {p}"
    print(f"  [ok] {p}  ({p.stat().st_size:,} bytes)")

# Print the manifest's headline info.
manifest = json.loads((out / "run_manifest.json").read_text())
print()
print("Run manifest:")
print(f"  timestamp:    {manifest['timestamp_utc']}")
print(f"  git commit:   {manifest['git_commit']}")
print(f"  python:       {manifest['python']}")
print(f"  torch:        {manifest['torch']}")
print(f"  cuda:         {manifest['cuda_available']}")
print(f"  gpu:          {manifest['gpu_name']}")
print(f"  folds run:    {[fr['fold'] for fr in manifest['fold_results']]}")
print()
print("Done. Now click 'Save Version → Save & Run All (Commit)' to persist "
      "these outputs. After the run completes, download:")
print("  • predictions.parquet")
print("  • run_manifest.json")
print("  • (optional) train_metrics.json, checkpoints/")
print()
print("Place them in artifacts/ locally, then run `python compare.py`.")
